## 1. Setup & MNIST Download

Downloads the raw MNIST files, normalizes pixel values to `[0, 1]`, encodes labels as one-hot vectors, and writes two packed `f32` binary files (`data/mnist_train.bin`, `data/mnist_test.bin`) in the format expected by O.N.O (`x_size=784`, `y_size=10`, 794 floats per row).

In [5]:
import os, struct, gzip, shutil, urllib.request, subprocess, time
import numpy as np

NWORKERS = 3
NSERVERS = 2
BASE_WORKER_PORT = 50000
BASE_SERVER_PORT = 40000
WORKER_ADDRS = [f"127.0.0.1:{BASE_WORKER_PORT + i}" for i in range(NWORKERS)]
SERVER_ADDRS = [f"127.0.0.1:{BASE_SERVER_PORT + i}" for i in range(NSERVERS)]

URLS = {
    "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "test_images":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
}
RAW_DIR  = "data/mnist_raw"
TRAIN_BIN = "data/mnist_train.bin"
TEST_BIN  = "data/mnist_test.bin"
SAFETENSORS_PATH = "data/mnist_model.safetensors"
X_SIZE, Y_SIZE, ROW_SIZE = 784, 10, 794

def download(name, url):
    os.makedirs(RAW_DIR, exist_ok=True)
    gz, raw = os.path.join(RAW_DIR, name + ".gz"), os.path.join(RAW_DIR, name)
    if os.path.exists(raw):
        print(f"  already exists: {raw}"); return raw
    print(f"  downloading {name}...")
    urllib.request.urlretrieve(url, gz)
    with gzip.open(gz, "rb") as fi, open(raw, "wb") as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(gz)
    return raw

def read_images(path):
    with open(path, "rb") as f:
        _, n, rows, cols = struct.unpack(">IIII", f.read(16))
        raw = f.read(n * rows * cols)
    ppi = rows * cols
    return [[b / 255.0 for b in raw[i*ppi:(i+1)*ppi]] for i in range(n)]

def read_labels(path):
    with open(path, "rb") as f:
        _, n = struct.unpack(">II", f.read(8))
        return list(f.read(n))

def write_bin(images, labels, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "wb") as f:
        for img, lbl in zip(images, labels):
            oh = [0.0] * Y_SIZE; oh[lbl] = 1.0
            f.write(struct.pack(f"{ROW_SIZE}f", *(img + oh)))
    print(f"  wrote {len(images)} samples → {out_path} ({os.path.getsize(out_path)/1024/1024:.1f} MB)")

for name, url in URLS.items():
    download(name, url)

if not os.path.exists(TRAIN_BIN):
    write_bin(read_images(os.path.join(RAW_DIR,"train_images")), read_labels(os.path.join(RAW_DIR,"train_labels")), TRAIN_BIN)
else:
    print(f"training set already exists: {TRAIN_BIN}")

if not os.path.exists(TEST_BIN):
    write_bin(read_images(os.path.join(RAW_DIR,"test_images")), read_labels(os.path.join(RAW_DIR,"test_labels")), TEST_BIN)
else:
    print(f"test set already exists: {TEST_BIN}")

  already exists: data/mnist_raw/train_images
  already exists: data/mnist_raw/train_labels
  already exists: data/mnist_raw/test_images
  already exists: data/mnist_raw/test_labels
training set already exists: data/mnist_train.bin
test set already exists: data/mnist_test.bin


## 2. Start Workers & Servers

Spawns `NSERVERS` parameter servers and `NWORKERS` workers as background subprocesses using `cargo run --release`. Each process logs to `temp/`. A short sleep is added between servers and workers to ensure servers are listening before workers try to connect.

In [ ]:
procs = []
os.makedirs("temp", exist_ok=True)

print(f"Starting {NSERVERS} server(s)...")
for i in range(NSERVERS):
    port = BASE_SERVER_PORT + i
    log = open(f"temp/server-{i}.log", "w")
    p = subprocess.Popen(
        ["cargo", "run", "--release", "-p", "parameter_server"],
        env={**os.environ, "PORT": str(port), "RUST_LOG": "info"},
        stdout=log, stderr=log,
    )
    procs.append((f"server-{i}", p, log))
    print(f"  server-{i} → port {port} (pid {p.pid})")

time.sleep(1.5)

print(f"Starting {NWORKERS} worker(s)...")
for i in range(NWORKERS):
    port = BASE_WORKER_PORT + i
    log = open(f"temp/worker-{i}.log", "w")
    p = subprocess.Popen(
        ["cargo", "run", "--release", "-p", "worker"],
        env={**os.environ, "PORT": str(port), "RUST_LOG": "info"},
        stdout=log, stderr=log,
    )
    procs.append((f"worker-{i}", p, log))
    print(f"  worker-{i} → port {port} (pid {p.pid})")

time.sleep(1.5)
print("All entities running.")

Starting 2 server(s)...
  server-0 → port 40000 (pid 215755)
  server-1 → port 40001 (pid 215756)


## 3. Distributed Training with O.N.O

Builds a 784→128→64→10 dense network with Kaiming initialization and Sigmoid activations, then trains it distributedly using the Parameter Server algorithm (`NonBlockingSync`, `WildStore`, batch size 256, 300 epochs). Once complete, the trained weights are saved to `data/mnist_model.safetensors`.

In [ ]:
import orchestra
from orchestra import Sequential, orchestrate
from orchestra.arch import Dense
from orchestra.activations import Sigmoid
from orchestra.initialization import Kaiming
from orchestra.datasets import LocalDataset
from orchestra.optimizers import GradientDescent
from orchestra.sync import NonBlockingSync
from orchestra.store import WildStore

model = Sequential([
    Dense(128, Kaiming(), Sigmoid()),
    Dense(64,  Kaiming(), Sigmoid()),
    Dense(10,  Kaiming()),
])

training = orchestra.parameter_server(
    worker_addrs=WORKER_ADDRS,
    server_addrs=SERVER_ADDRS,
    dataset=LocalDataset(TRAIN_BIN, x_size=784, y_size=10),
    optimizer=GradientDescent(lr=0.01),
    sync=NonBlockingSync(),
    store=WildStore(),
    max_epochs=100,
    batch_size=256,
    seed=42,
)

print("Starting training session...")
session = orchestrate(model, training)
trained = session.wait()
print(f"Training complete — {len(trained.weights())} parameters")

trained.save_safetensors(SAFETENSORS_PATH)
size = os.path.getsize(SAFETENSORS_PATH)
assert size > 0
print(f"Model saved to {SAFETENSORS_PATH} ({size} bytes)")

Starting training session...


  ⠋ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 1/300
  avg_loss=0.10221042
  ⠙ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 1/300
  avg_loss=0.10222504
  ⠹ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 1/300
  avg_loss=0.10217010
  ⠸ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 2/300
  avg_loss=0.11110058
  ⠼ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 2/300
  avg_loss=0.11995264
  ⠴ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 2/300
  avg_loss=0.12883501
  ⠦ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 3/300
  avg_loss=0.16990586
  ⠧ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 3/300
  avg_loss=0.21084221
  ⠇ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 3/300
  avg_loss=0.25181016
  ⠏ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 4/300
  avg_loss=0.27079043
  ⠋ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 4/300
  avg_loss=0.28958377
  ⠙ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 4/300
  avg_loss=0.30881628
  ⠹ [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 5/300
  avg_loss=0.413

KeyboardInterrupt: 

## 4. Accuracy Evaluation with PyTorch

Loads the saved `.safetensors` file into an equivalent PyTorch model. O.N.O stores weight matrices as `[input, output]` so they are transposed before loading into PyTorch's `Linear` layers (which expect `[output, input]`). The full 10,000-sample MNIST test set is then evaluated and top-1 accuracy is printed.

In [ ]:
import torch
import torch.nn as nn
from safetensors.torch import load_file

state_dict = load_file(SAFETENSORS_PATH)
print("Loaded tensors:", list(state_dict.keys()))

class MnistNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc0 = nn.Linear(784, 128)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64,  10)

    def forward(self, x):
        x = torch.sigmoid(self.fc0(x))
        x = torch.sigmoid(self.fc1(x))
        return self.fc2(x)

net = MnistNet()

with torch.no_grad():
    net.fc0.weight.copy_(state_dict["layer_0.weight"].T)
    net.fc0.bias.copy_(state_dict["layer_0.bias"])
    net.fc1.weight.copy_(state_dict["layer_1.weight"].T)
    net.fc1.bias.copy_(state_dict["layer_1.bias"])
    net.fc2.weight.copy_(state_dict["layer_2.weight"].T)
    net.fc2.bias.copy_(state_dict["layer_2.bias"])

net.eval()

raw = np.fromfile(TEST_BIN, dtype=np.float32).reshape(-1, ROW_SIZE)
x_test = torch.tensor(raw[:, :X_SIZE])
y_test = torch.tensor(raw[:, X_SIZE:].argmax(axis=1), dtype=torch.long)

with torch.no_grad():
    preds = net(x_test).argmax(dim=1)
    acc   = (preds == y_test).float().mean().item() * 100

print(f"\nTest accuracy: {acc:.2f}%")

Loaded tensors: ['layer_0.bias', 'layer_0.weight', 'layer_1.bias', 'layer_1.weight', 'layer_2.bias', 'layer_2.weight']

Test accuracy: 46.17%


## 5. Stop Workers & Servers

Terminates all worker and server subprocesses started in step 2 and closes their log file handles.

In [ ]:
print("Stopping all entities...")
for name, p, log in procs:
    p.terminate()
    p.wait(timeout=5)
    log.close()
    print(f"  stopped {name} (pid {p.pid})")
print("Done.")

Stopping all entities...
  stopped server-0 (pid 209846)
  stopped server-1 (pid 209847)
  stopped worker-0 (pid 209895)
  stopped worker-1 (pid 209896)
  stopped worker-2 (pid 209897)
Done.
